# BERT Model Experiments

This notebook prepares and fine-tunes a pretrained BERT model to classify crisis-related social media posts into 11 humanitarian categories. It uses the shared preprocessed datasets from `data/processed/`, with `text_clean` as the model input and `class_label` as the target. Model performance will be monitored on the validation set, with the test set reserved for final evaluation.

In [3]:
#imports
import sys
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Make the project root available so Python can import src.bert_model.
project_root = Path(
    "/Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp"
)

if not (project_root / "src" / "bert_model.py").exists():
    raise FileNotFoundError(
        f"Could not find src/bert_model.py in {project_root}"
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


/Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Project root: /Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp


In [4]:
from src.bert_model import load_data, prepare_labels, tokenize_data

# Load the shared, preprocessed datasets.
train_df, dev_df, test_df = load_data()

# Create a consistent numerical mapping for the 11 categories.
train_df, dev_df, test_df, label2id, id2label = prepare_labels(
    train_df, dev_df, test_df
)

print("Training examples:", f"{len(train_df):,}")
print("Validation examples:", f"{len(dev_df):,}")
print("Test examples:", f"{len(test_df):,}")
print("Number of categories:", len(label2id))

print("\nTraining label counts:")
print(train_df["class_label"].value_counts())


Training examples: 61,164
Validation examples: 8,934
Test examples: 17,356
Number of categories: 11

Training label counts:
class_label
not_humanitarian                       36071
donation_and_volunteering               5175
requests_or_needs                       4829
sympathy_and_support                    3545
infrastructure_and_utilities_damage     3539
affected_individual                     2452
caution_and_advice                      2100
injured_or_dead_people                  1942
response_efforts                         780
missing_and_found_people                 373
displaced_and_evacuations                358
Name: count, dtype: int64


## Initialize BERT

Load the pretrained BERT tokenizer and classification model. The classifier is configured for the project's 11 categories and uses Apple GPU acceleration when available.

In [5]:
#initialize the model
MODEL_NAME = "bert-base-uncased"
NUM_LABELS = len(label2id)
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)
model.to(device)

print("Model initialized successfully.")
print("Device:", device)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model initialized successfully.
Device: mps


### Prepare Text and Labels

BERT requires text to be converted into tokens and labels to be represented numerically. This section creates a consistent mapping for the 11 humanitarian categories and tokenizes the training, validation, and test datasets.

In [6]:
# Create a consistent mapping of category names to numeric labels
label_names = sorted(train_df["class_label"].unique())

label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

# Convert category names to numeric labels
train_df["label"] = train_df["class_label"].map(label2id)
dev_df["label"] = dev_df["class_label"].map(label2id)
test_df["label"] = test_df["class_label"].map(label2id)

print("Number of classes:", len(label_names))
print("\nLabel mapping:")
print(label2id)

Number of classes: 11

Label mapping:
{'affected_individual': 0, 'caution_and_advice': 1, 'displaced_and_evacuations': 2, 'donation_and_volunteering': 3, 'infrastructure_and_utilities_damage': 4, 'injured_or_dead_people': 5, 'missing_and_found_people': 6, 'not_humanitarian': 7, 'requests_or_needs': 8, 'response_efforts': 9, 'sympathy_and_support': 10}


In [7]:
#tokenize and test

# Tokenize text_clean and create PyTorch datasets.
train_dataset, dev_dataset, test_dataset = tokenize_data(
    train_df, dev_df, test_df, tokenizer
)

# Verify dataset sizes and label consistency.
assert len(train_dataset) == len(train_df)
assert len(dev_dataset) == len(dev_df)
assert len(test_dataset) == len(test_df)
assert len(label2id) == 11

for name, df in [
    ("training", train_df),
    ("validation", dev_df),
    ("test", test_df),
]:
    assert df["label"].notna().all(), f"Missing labels in {name}"
    assert set(df["class_label"]) == set(label2id), (
        f"Unexpected categories in {name}"
    )

# Check one example.
example = train_dataset[0]
assert example["input_ids"].shape[0] == MAX_LENGTH
assert example["attention_mask"].shape[0] == MAX_LENGTH

print("All tests passed!")
print(f"Training examples: {len(train_dataset):,}")
print(f"Validation examples: {len(dev_dataset):,}")
print(f"Test examples: {len(test_dataset):,}")
print(f"Number of categories: {len(label2id)}")
print("Model input: text_clean")
print("Target: class_label")
print("Device:", device)

print("\nExample input shape:", example["input_ids"].shape)
print("Example numeric label:", example["labels"].item())
print("Example category:", id2label[example["labels"].item()])


All tests passed!
Training examples: 61,164
Validation examples: 8,934
Test examples: 17,356
Number of categories: 11
Model input: text_clean
Target: class_label
Device: mps

Example input shape: torch.Size([128])
Example numeric label: 4
Example category: infrastructure_and_utilities_damage


In [10]:
import importlib
import src.bert_model

importlib.reload(src.bert_model)

from src.bert_model import train_model

In [11]:
# Train and evaluate BERT on a small sample first
model, training_history = train_model(
    train_dataset,
    dev_dataset,
    label2id,
    id2label,
)
history_df = pd.DataFrame(training_history)
display(history_df)

Training device: mps
Training examples: 61164
Validation examples: 8934


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3: 100/7646 training batches completed
Epoch 1/3: 200/7646 training batches completed
Epoch 1/3: 300/7646 training batches completed
Epoch 1/3: 400/7646 training batches completed
Epoch 1/3: 500/7646 training batches completed
Epoch 1/3: 600/7646 training batches completed
Epoch 1/3: 700/7646 training batches completed
Epoch 1/3: 800/7646 training batches completed
Epoch 1/3: 900/7646 training batches completed
Epoch 1/3: 1000/7646 training batches completed
Epoch 1/3: 1100/7646 training batches completed
Epoch 1/3: 1200/7646 training batches completed
Epoch 1/3: 1300/7646 training batches completed
Epoch 1/3: 1400/7646 training batches completed
Epoch 1/3: 1500/7646 training batches completed
Epoch 1/3: 1600/7646 training batches completed
Epoch 1/3: 1700/7646 training batches completed
Epoch 1/3: 1800/7646 training batches completed
Epoch 1/3: 1900/7646 training batches completed
Epoch 1/3: 2000/7646 training batches completed
Epoch 1/3: 2100/7646 training batches completed
E

,epoch,train_loss,val_macro_f1
0,1,0.926870,0.705585
1,2,0.526442,0.736039
2,3,0.336542,0.741592


In [13]:
from pathlib import Path
import shutil

project_root = Path.cwd().parent  # if your notebook is in notebooks/
source = Path.cwd() / "models" / "bert_crisis_classifier"
destination = project_root / "models" / "bert_crisis_classifier"

destination.parent.mkdir(parents=True, exist_ok=True)

if source.resolve() != destination.resolve():
    shutil.copytree(source, destination, dirs_exist_ok=True)

print(f"Model files are at: {destination.resolve()}")

Model files are at: /Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp/models/bert_crisis_classifier
